# Simulated PAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 1  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4 #low-frequency rhythm (theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_phase = np.angle(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate amp_mod(t) from word onsets and kernel ---
word_onsets = np.zeros(n_samples)
#word_locs = np.random.randint(100, 900, size=5)  # random word onset indices
word_locs = np.random.randint(100, 900, size=1000)
word_onsets[word_locs] = 1
# 100ms - 3000sec

# Gaussian kernel: spreads effect of each word over ~100 ms
kernel_width = int(0.1 * sfreq)  # 100 ms
kernel = np.exp(-np.linspace(-1, 1, 2 * kernel_width)**2 / 0.05)
kernel = kernel / kernel.max()  # normalize to [0, 1]

# Convolve to get slow modulation function
amp_mod = np.convolve(word_onsets, kernel, mode='same')
amp_mod = 0.25 + 0.75 * (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2*np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope represents independent background amplitude fluctuation in the gamma band.


# --- Step 3: Final high-frequency envelope ---
# Implements A_H(t) = (1 - mod) * hf_amp_ind + mod * f(φ_L(t))
sigma = np.pi / 2
f_phi_l = np.exp(-(lf_phase ** 2) / (2 * sigma ** 2))  # f(φ_L(t))
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l


# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
# plt.plot(times, lf_signal, label='lf_signal', linestyle ='--')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
# plt.plot(times, lf_phase, label='lf_phase(t)', linestyle=':')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.plot(times, f_phi_l, label='f(ϕ_L(t))', linestyle=':')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()





In [ ]:
print(times[word_locs[:10]])  # Print first few word onset times in seconds


# Test mTRF on hf_envelope

X1 = f_phi_l: phase-based predictor (1D)

X2 = amp_mod: strength modulator (1D)

X3 = interaction = f_phi_l * amp_mod: interaction term

Y = hf_envelope: the ground-truth PAC envelope

In [ ]:
from sklearn.linear_model import Ridge
from scipy.linalg import toeplitz

def make_lagged_matrix(x, lags):
    """Construct lagged predictor matrix with given lags."""
    n = len(x)
    lagged = np.stack([np.roll(x, -lag) for lag in lags], axis=1)
    lagged[:lags[-1], :] = 0  # zero-padding
    return lagged

sfreq = 1000  # 1 kHz
tmin, tmax = -0.1, 0.3  # 100 ms before to 300 ms after
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq))  # e.g., -100 to +299 ms

X_lagged = np.hstack([
    make_lagged_matrix(f_phi_l, lags),
    make_lagged_matrix(amp_mod, lags),
    make_lagged_matrix(f_phi_l * amp_mod, lags)
])

ridge = Ridge(alpha=1.0)  # regularization strength
ridge.fit(X_lagged, hf_envelope)
w = ridge.coef_  # shape: (n_lags * n_predictors,)

trf_f_phi = w[:len(lags)]
trf_amp_mod = w[len(lags):2*len(lags)]
trf_interact = w[2*len(lags):]

time_lags_ms = lags * 1000 / sfreq

plt.figure(figsize=(10, 4))
plt.plot(time_lags_ms, trf_f_phi, label='TRF: f(ϕ_L(t))')
plt.plot(time_lags_ms, trf_amp_mod, label='TRF: amp_mod(t)')
plt.plot(time_lags_ms, trf_interact, label='TRF: interaction')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("mTRF Weights for PAC Simulation")
plt.legend()
plt.tight_layout()
plt.show()


The mTRF recovered distinct temporal response functions for:

* the low-frequency phase function
𝑓
(
𝜙
𝐿
(
𝑡
))

* the speech-modulated coupling term
amp_mod
(
𝑡
), and

* their interaction, which encodes true phase–amplitude coupling (PAC).

Only the interaction term shows a large, temporally specific peak near 0–100 ms lag, validating the simulation and demonstrating that mTRF can detect modulated PAC structures in synthetic EEG.

# permutation testing
We want to know:

Are the mTRF weights learned from f(ϕ_L(t)) × amp_mod(t) significantly better than chance?

We'll test this by:

Shuffling amp_mod(t) (or f_phi_l) to break the PAC structure

Refitting mTRFs with shuffled predictors

Building a null distribution of TRF weights

Comparing your real TRF weights against this distribution



In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

def make_lagged_matrix(x, lags):
    return np.stack([np.roll(x, -lag) for lag in lags], axis=1)

# --- Original predictors ---
# use word onset as predictor
X_fphi = make_lagged_matrix(f_phi_l, lags)
X_onset = make_lagged_matrix(word_onsets, lags)
# X_amp = make_lagged_matrix(amp_mod, lags)
X_inter = make_lagged_matrix(f_phi_l * word_onsets, lags)

# --- Target ---

Y = hf_envelope

# --- Fit real model ---
X_real = np.hstack([X_fphi, X_amp, X_inter])
ridge = Ridge(alpha=1.0)
ridge.fit(X_real, Y)
w_real = ridge.coef_

# --- Permutation test ---
n_permutations = 1000
w_nulls = np.zeros((n_permutations, len(w_real)))

for i in range(n_permutations):
    # Shuffle amp_mod (can also shuffle f_phi_l if preferred)
    amp_mod_shuffled = np.random.permutation(amp_mod)
    X_amp_shuff = make_lagged_matrix(amp_mod_shuffled, lags)
    X_inter_shuff = make_lagged_matrix(f_phi_l * amp_mod_shuffled, lags)

    X_null = np.hstack([X_fphi, X_amp_shuff, X_inter_shuff])

    model_null = Ridge(alpha=1.0)
    model_null.fit(X_null, Y)
    w_nulls[i, :] = model_null.coef_

# --- Compute p-values ---
from scipy.stats import percentileofscore

# p-values for interaction term
real_inter = w_real[2 * len(lags):]
null_inter = w_nulls[:, 2 * len(lags):]

p_vals = np.array([
    1 - (percentileofscore(null_inter[:, j], real_inter[j]) / 100)
    for j in range(len(lags))
])


plt.figure(figsize=(10, 4))
plt.plot(lags * 1000 / sfreq, real_inter, label='Real TRF: interaction', color='green')
plt.fill_between(
    lags * 1000 / sfreq, 0, real_inter,
    where=p_vals < 0.05, color='red', alpha=0.3, label='p < 0.05'
)
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("Significant TRF Weights for PAC Interaction")
plt.legend()
plt.tight_layout()
plt.show()


We used permutation-based testing (N=1000) to assess significance of the mTRF weights corresponding to the PAC interaction regressor.
A significant peak was observed at ~0 ms lag (p < 0.05), indicating reliable recovery of the induced PAC mechanism in our simulated EEG.

# Test mTrf on hf_signal


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from scipy.stats import percentileofscore
import matplotlib.pyplot as plt

# === Step 1: Define lags ===
sfreq = 1000  # Sampling frequency
tmin, tmax = -0.1, 0.3  # 100 ms before to 300 ms after
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq))

def make_lagged_matrix(x, lags):
    return np.stack([np.roll(x, -lag) for lag in lags], axis=1)

# === Step 2: Create lagged predictors ===
X_fphi = make_lagged_matrix(f_phi_l, lags)
# X_amp = make_lagged_matrix(amp_mod, lags)
# X_inter = make_lagged_matrix(f_phi_l * amp_mod, lags)
X_onset = make_lagged_matrix(word_onsets, lags)
# treat word onset as a trial
X_inter = make_lagged_matrix(f_phi_l * word_onsets, lags)

# === Step 3: Stack into full predictor matrix ===
# X = np.hstack([X_fphi, X_amp, X_inter])
X = np.hstack([X_fphi, X_onset, X_inter])

# === Step 4: Define target as hf_signal (not envelope) ===
# complex decomposition of hf_signal
Y = hf_signal

# === Step 5: Fit mTRF model ===
ridge = Ridge(alpha=1.0)
ridge.fit(X, Y)
w_real = ridge.coef_

# === Step 6: Extract TRF for interaction term ===
w_inter = w_real[2 * len(lags):]  # last third of the weights

# Compute p-values
p_vals = np.array([
    1 - (percentileofscore(w_nulls[:, j], w_inter[j]) / 100)
    for j in range(len(lags))
])

# Plot
plt.figure(figsize=(10, 4))
plt.plot(lags * 1000 / sfreq, w_inter, label='TRF: interaction (raw EEG)', color='green')
plt.fill_between(lags * 1000 / sfreq, 0, w_inter,
                 where=p_vals < 0.05, color='red', alpha=0.3, label='p < 0.05')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("Significant TRF Weights from Raw EEG (PAC Interaction)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

# === Lowpass filter parameters ===
sfreq = 1000  # Hz
cutoff = 10   # Hz (preserve slow modulations only)
order = 4

# === Design and apply filter ===
b, a = butter(order, cutoff / (sfreq / 2), btype='low')
hf_lowpassed = filtfilt(b, a, hf_signal)

Y = hf_lowpassed

# === Step 5: Fit mTRF model ===
ridge = Ridge(alpha=1.0)
ridge.fit(X, Y)
w_real = ridge.coef_

# === Step 6: Extract TRF for interaction term ===
w_inter = w_real[2 * len(lags):]  # last third of the weights

# Compute p-values
p_vals = np.array([
    1 - (percentileofscore(w_nulls[:, j], w_inter[j]) / 100)
    for j in range(len(lags))
])

# Plot
plt.figure(figsize=(10, 4))
plt.plot(lags * 1000 / sfreq, w_inter, label='TRF: interaction (raw EEG)', color='green')
plt.fill_between(lags * 1000 / sfreq, 0, w_inter,
                 where=p_vals < 0.05, color='red', alpha=0.3, label='p < 0.05')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("Significant TRF Weights from Raw EEG (PAC Interaction)")
plt.legend()
plt.tight_layout()
plt.show()